# BaseKit 网络接口调试指南 (纯 Kotlin/Java 实现)

在 Kotlin Notebook 中，由于缺少 Android 运行环境及项目的自动依赖解析，直接引入 `BaseKit` 源码类（如 `KCHttp`）会报 `Unresolved reference`。

因此，下面提供了一套 **纯 Kotlin/Java 系统类** 的实现方案，不需要任何外部依赖，可以直接在 Notebook 中运行，用来模拟和调试日常接口。

## 1. 发起普通的 GET 请求
利用 Kotlin 对 `java.net.URL` 的扩展函数 `readText()`，只需一行代码即可发起 GET 请求。

In [ ]:
import java.net.URL

try {
    val url = "https://api.github.com/zen"
    val response = URL(url).readText()
    println("GET 请求成功:\n$response")
} catch (e: Exception) {
    println("GET 请求失败: ${e.message}")
}

## 2. 发起 POST JSON 请求
使用 `java.net.HttpURLConnection` 构建一个标准的 POST 请求，并发送 JSON Body。

In [3]:
import java.net.URL
import java.net.HttpURLConnection

try {
    val url = URL("https://jsonplaceholder.typicode.com/posts")
    val jsonBody = """{"title":"BaseKit","body":"Testing pure Kotlin POST","userId":1}"""
    
    with(url.openConnection() as HttpURLConnection) {
        requestMethod = "POST"
        doOutput = true
        setRequestProperty("Content-Type", "application/json; utf-8")
        setRequestProperty("Accept", "application/json")
        
        // 写入请求体
        outputStream.write(jsonBody.toByteArray(Charsets.UTF_8))
        
        // 读取响应结果
        val response = inputStream.bufferedReader().readText()
        println("POST 请求成功:\n$response")
    }
} catch (e: Exception) {
    println("POST 请求失败: ${e.message}")
}

POST 请求成功:
{
  "id": 101
}


## 3. 文件下载
通过 `URL.openStream()` 获取输入流，并使用 Kotlin 的 `copyTo` 直接写入到本地文件中。

In [2]:
import java.net.URL
import java.io.File

try {
    val url = URL("https://raw.githubusercontent.com/shetj/BaseKit/master/README.md")
    val saveFile = File("./README_DOWNLOAD_TEST.md")
    
    url.openStream().use { input ->
        saveFile.outputStream().use { output ->
            input.copyTo(output)
        }
    }
    println("下载完成，文件保存在: ${saveFile.absolutePath}")
    println("文件大小: ${saveFile.length()} bytes")
} catch (e: Exception) {
    println("下载失败: ${e.message}")
}

下载失败: Connection reset


## 4. SSE (Server-Sent Events) 连接
借助 Java 11 引入的 `java.net.http.HttpClient`，可以非常方便地处理 SSE 的数据流。由于是在 Notebook 中运行，这里使用异步任务并通过 `limit(5)` 只读取前 5 条消息后自动断开，避免阻塞整个 Notebook 进程。

In [ ]:
import java.net.URI
import java.net.http.HttpClient
import java.net.http.HttpRequest
import java.net.http.HttpResponse

try {
    val client = HttpClient.newHttpClient()
    val request = HttpRequest.newBuilder()
        .uri(URI.create("https://express-eventsource.herokuapp.com/events"))
        .header("Accept", "text/event-stream")
        .GET()
        .build()
    
    println("开始连接 SSE...")
    
    // 异步发送请求，并按行处理返回的流
    client.sendAsync(request, HttpResponse.BodyHandlers.ofLines())
        .thenAccept { response ->
            val stream = response.body()
            // 读取前 5 条有效消息后停止
            stream.limit(5).forEach { line ->
                if (line.isNotBlank()) {
                    println("收到 SSE 消息: $line")
                }
            }
            println("SSE 演示结束")
        }
        .join() // 阻塞等待异步执行完成，以便在 Notebook 中看到输出
        
} catch (e: Exception) {
    println("SSE 连接失败: ${e.message}")
}

## 5. 执行 Curl 命令
Kotlin 允许通过 `Runtime.getRuntime().exec()` 或 `ProcessBuilder` 执行系统命令。通过这种方式，我们可以在 Notebook 中直接执行并获取 `curl` 的返回结果。

In [ ]:
import java.io.BufferedReader
import java.io.InputStreamReader

try {
    // 构建一个简单的 curl 命令，例如获取本机的出口 IP 或者测试接口
    val command = listOf("curl", "-s", "https://httpbin.org/get")
    
    // 使用 ProcessBuilder 执行命令
    val processBuilder = ProcessBuilder(command)
    processBuilder.redirectErrorStream(true)
    val process = processBuilder.start()
    
    // 读取命令的输出
    val reader = BufferedReader(InputStreamReader(process.inputStream))
    val output = StringBuilder()
    var line: String?
    while (reader.readLine().also { line = it } != null) {
        output.append(line).append("\n")
    }
    
    // 等待命令执行完成
    val exitCode = process.waitFor()
    println("命令执行完成，退出码: $exitCode")
    println("返回结果:\n$output")
} catch (e: Exception) {
    println("执行 curl 命令失败: ${e.message}")
}